In [ ]:
# Fine-tune YOLOX-Nano on gray footage from your own sites, with Sharpness-Aware Minimization, and accept the
# result only if MCC on held-out frames says it is better than the released model. One cell.
#
#   1. YOLOX-Nano's released weights, folded to take 1 channel (exactly: the first convolution's weights summed
#      over the colour channels), checked against the 3-channel model.
#   2. Your labelled frames, read as gray (COCO JSON, or YOLO txt labels), split into train and held-out.
#   3. Training with SAM (Foret et al. 2021) around SGD, cosine learning rate, gray-picture augmentation.
#   4. The acceptance test: per held-out frame and per class, "is there one?" decided at DECISION_SCORE, scored
#      with the Matthews correlation coefficient against the labels, for the released model and the new one.
#      The new model passes only if its MCC beats the released model's by MIN_GAIN, reaches MIN_MCC, and no
#      class loses more than MAX_CLASS_DROP.
#   5. Only a passing model is exported: ONNX, 1 channel, any batch, in the released model's output format, so
#      `tamserve-boxes` and `tamserve.boxes` run it as they are (set detector.names if the classes changed).
#
# Point DATA at your footage. Left as it is, it downloads coco128 (128 COCO pictures, GPL-3.0 as Ultralytics
# packages them) only to show the pipeline working: that is not a test of your sites.
import hashlib, json, math, os, random, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

DATA = {
    "format": "yolo",                # "coco": one JSON per split; "yolo": images/ and labels/ with class cx cy w h
    "train": None, "val": None,      # coco: {"images": dir, "annotations": file.json}; yolo: {"images": dir, "labels": dir}
    "split": 0.2,                    # with no "val", this share of the frames (whole sites/cameras ideally) is held out
    "names": None,                   # yolo: what each class number in the label files means; None: COCO's 80
    "keep": None,                    # the classes to train on; other labels are dropped. None: all of them. Keep every
                                     # class your labels mark completely: a dropped label teaches "that is background"
}                                    # COCO classes keep the released head; any other name gets a new one
EPOCHS, BATCH, LR, RHO, SIZE = 10, 8, 0.0001, 0.05, 416  # RHO: SAM's neighbourhood radius
MOMENTUM = 0.0                      # YOLOX-Nano with frozen BatchNorm is sharp: momentum 0.9 compounds the step about
                                    # tenfold and diverged at LR 1e-4; plain SGD at 1e-4 lowers the loss steadily
EMA_DECAY = 0.99                    # the gate and the export use an average of the weights, as YOLOX's own training does
CLIP = 10.0                         # gradient norm clip: a few bad frames must not throw the weights off
FREEZE_BN = True                    # keep BatchNorm's statistics as released: batches of 8 are too small to re-estimate them
DECISION_SCORE = 0.5                # the confidence at which a rule would say "there is one"
MIN_GAIN, MIN_MCC, MAX_CLASS_DROP = 0.0, 0.0, 0.05
REQUIRE_SURE_GAIN = False           # also require the bootstrap 90% interval of the MCC gain to be above 0
EVAL_CLASSES = None                 # classes the gate scores (your rules' classes); None: those in the held-out labels
SEED = 0
# Demo only: turn the demonstration pictures into CCTV-like ones (half resolution, blur, dark, sensor noise, heavy
# JPEG), so the released model meets footage unlike what it was trained on, as on a real site. Ignored for your data.
DEMO_CCTV = True
_env = os.environ.get("FT_OVERRIDE")  # for testing this cell: JSON of settings to override
if _env:
    globals().update(json.loads(_env))

WORK = Path("/content" if Path("/content").exists() else Path.cwd()) / "finetune"
WORK.mkdir(parents=True, exist_ok=True)
YOLOX_COMMIT = "6ddff4824372906469a7fae2dc3206c7aa4bbaee"

def sh(*a, **k):
    return subprocess.run(list(a), check=True, **k)

def fetch(url, sha256, name=None):
    path = WORK / (name or url.rsplit("/", 1)[1])
    if not path.exists():
        data = urllib.request.urlopen(url, timeout=300).read()
        assert sha256 is None or hashlib.sha256(data).hexdigest() == sha256, f"{url} is not the pinned file"
        path.write_bytes(data)
    return path

if not (WORK / "YOLOX").exists():
    sh("git", "clone", "-q", "https://github.com/Megvii-BaseDetection/YOLOX", str(WORK / "YOLOX"))
    sh("git", "-C", str(WORK / "YOLOX"), "checkout", "-q", YOLOX_COMMIT)
sh(sys.executable, "-m", "pip", "install", "-q", "loguru", "tabulate", "thop", "packaging", "onnx", "openvino>=2024.0",
   "opencv-python-headless", "pycocotools")
sys.path.insert(0, str(WORK / "YOLOX"))

import cv2, numpy as np, torch, torch.nn as nn, torchvision
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
COCO = ["person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
        "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
        "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
        "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
        "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
        "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
        "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard",
        "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
        "teddy bear", "hair drier", "toothbrush"]
LABEL_NAMES = DATA["names"] or COCO
KEEP = DATA["keep"]
SAME_HEAD = all(c in COCO for c in (KEEP or LABEL_NAMES))  # COCO classes keep the released head; new ones get a new head
HEAD_ROWS = COCO if SAME_HEAD else list(KEEP or LABEL_NAMES)
kept = lambda name: name in HEAD_ROWS and (KEEP is None or name in KEEP)

# ---------------------------------------------------------------------------------------------- 1. the model
def nano(num_classes):
    ch = [256, 512, 1024]
    m = YOLOX(YOLOPAFPN(0.33, 0.25, in_channels=ch, act="silu", depthwise=True),
              YOLOXHead(num_classes, 0.25, in_channels=ch, act="silu", depthwise=True))
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.eps, mod.momentum = 1e-3, 0.03
    return m

def fold_to_gray(m):
    """The first convolution reads Focus's 4 patches x 3 colours; summed over the colours it reads 4, exactly."""
    conv = m.backbone.backbone.stem.conv.conv
    w = conv.weight.data
    folded = w.reshape(w.shape[0], w.shape[1] // 3, 3, *w.shape[2:]).double().sum(2).float()
    new = nn.Conv2d(folded.shape[1], w.shape[0], conv.kernel_size, conv.stride, conv.padding, bias=conv.bias is not None)
    new.weight.data.copy_(folded)
    m.backbone.backbone.stem.conv.conv = new
    return m

weights = fetch("https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
                "cd28f55fbbc1829f99d9ac9b38a16d259a22889739c8728ea877610201feff7b")
state = torch.load(weights, map_location="cpu", weights_only=False)["model"]
def released():
    m = nano(len(HEAD_ROWS))
    if SAME_HEAD:
        m.load_state_dict(state)
    else:  # everything but the class predictions, which are new
        keep = {k: v for k, v in state.items() if ".cls_preds." not in k}
        print("   new classes: the class layer starts from scratch;", m.load_state_dict(keep, strict=False).missing_keys[:3], "...")
    return m

rgb, gray_model = released().eval(), fold_to_gray(released()).eval()
for m in (rgb, gray_model):
    m.head.decode_in_inference = False
g = torch.randint(0, 256, (2, 1, SIZE, SIZE)).float()
with torch.no_grad():
    d = (rgb(g.repeat(1, 3, 1, 1)) - gray_model(g)).abs().max().item()
print(f"1. 1-channel model vs 3-channel on gray input: largest difference {d:.1e} (float rounding)")

# ---------------------------------------------------------------------------------------------- 2. the data
def from_yolo(images, labels):
    out = []
    for p in sorted(Path(images).iterdir()):
        if p.suffix.lower() not in (".jpg", ".jpeg", ".png", ".bmp"):
            continue
        h, w = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).shape
        boxes = []
        lab = Path(labels) / (p.stem + ".txt")
        if lab.exists():
            for line in lab.read_text().split("\n"):
                if line.strip():
                    c, cx, cy, bw, bh = map(float, line.split()[:5])
                    name = LABEL_NAMES[int(c)]
                    if kept(name):
                        boxes.append((HEAD_ROWS.index(name), (cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
        out.append((str(p), boxes))
    return out

def from_coco(images, annotations):
    j = json.loads(Path(annotations).read_text())
    cats = {c["id"]: c["name"] for c in j["categories"]}
    per = {im["id"]: (str(Path(images) / im["file_name"]), []) for im in j["images"]}
    for a in j["annotations"]:
        name = cats[a["category_id"]]
        if kept(name) and not a.get("iscrowd"):
            x, y, w, h = a["bbox"]
            per[a["image_id"]][1].append((HEAD_ROWS.index(name), x, y, x + w, y + h))
    return list(per.values())

def read(split):
    return from_yolo(split["images"], split["labels"]) if DATA["format"] == "yolo" else from_coco(split["images"], split["annotations"])

DEMO_DATA = DATA["train"] is None
if DEMO_DATA:  # the demonstration data: scored on people, as a site's rules would be
    EVAL_CLASSES = EVAL_CLASSES or ["person"]
    z = fetch("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip", None)
    if not (WORK / "coco128").exists():
        zipfile.ZipFile(z).extractall(WORK)
    DATA["train"] = {"images": WORK / "coco128/images/train2017", "labels": WORK / "coco128/labels/train2017"}
frames = read(DATA["train"])
if DATA["val"] is not None:
    train, held = frames, read(DATA["val"])
else:
    random.Random(SEED).shuffle(frames)
    k = max(1, int(len(frames) * DATA["split"]))
    held, train = frames[:k], frames[k:]
print(f"2. {len(train)} training frames, {len(held)} held out; {sum(len(b) for _, b in train)} labelled boxes to learn from")

def letterbox(gray, size=SIZE):
    r = min(size / gray.shape[0], size / gray.shape[1])
    h, w = max(1, round(gray.shape[0] * r)), max(1, round(gray.shape[1] * r))
    out = np.full((size, size), 114, np.uint8)
    out[:h, :w] = cv2.resize(gray, (w, h), interpolation=cv2.INTER_LINEAR)
    return out, r

def cctv(g, rng):
    """Half resolution, a little blur, night-time gain, sensor noise, heavy JPEG: a camera at 2 am."""
    h, w = g.shape
    g = cv2.resize(cv2.resize(g, (w // 2, h // 2), interpolation=cv2.INTER_AREA), (w, h), interpolation=cv2.INTER_LINEAR)
    g = cv2.GaussianBlur(g, (0, 0), 1.0)
    g = 255 * (g / 255.0) ** 1.8 * 0.6
    g = np.clip(g + rng.normal(0, 7, g.shape), 0, 255).astype(np.uint8)
    return cv2.imdecode(cv2.imencode(".jpg", g, [cv2.IMWRITE_JPEG_QUALITY, 25])[1], cv2.IMREAD_GRAYSCALE)

def sample(path, boxes, train_mode):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if DEMO_CCTV and DEMO_DATA:  # the same degradation for a held-out frame every time; a fresh one for training
        g = cctv(g, np.random.default_rng(None if train_mode else int(hashlib.sha256(path.encode()).hexdigest()[:8], 16)))
    b = np.array([x[1:] for x in boxes], np.float32).reshape(-1, 4)
    c = np.array([x[0] for x in boxes], np.int64)
    if train_mode:
        s = random.uniform(0.6, 1.4)  # scale jitter
        g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_LINEAR); b *= s
        if random.random() < 0.5:     # mirror
            g = g[:, ::-1].copy(); b[:, [0, 2]] = g.shape[1] - b[:, [2, 0]]
        a, k = random.uniform(0.7, 1.3), random.uniform(-25, 25)  # contrast and brightness: lighting changes on site
        g = np.clip(g.astype(np.float32) * a + k, 0, 255).astype(np.uint8)
    img, r = letterbox(g)
    b *= r
    t = np.zeros((120, 5), np.float32)
    keep = ((b[:, 2] - b[:, 0]) > 2) & ((b[:, 3] - b[:, 1]) > 2)
    b, c = b[keep][:120], c[keep][:120]
    t[:len(b), 0] = c
    t[:len(b), 1:3] = (b[:, :2] + b[:, 2:]) / 2
    t[:len(b), 3:5] = b[:, 2:] - b[:, :2]
    return torch.from_numpy(img).float()[None], torch.from_numpy(t), r

# ---------------------------------------------------------------------------------------------- 3. SAM
class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimization: step to the worst point within RHO of the weights, take the gradient there,
    come back, and let the base optimizer step with it. Two forward-backward passes per step."""
    def __init__(self, params, base, rho=0.05, **kw):
        super().__init__(params, dict(rho=rho, **kw))
        self.base = base(self.param_groups, **kw)
        self.param_groups = self.base.param_groups

    @torch.no_grad()
    def first_step(self):
        norm = torch.norm(torch.stack([p.grad.norm(2) for g in self.param_groups for p in g["params"] if p.grad is not None]), 2)
        for g in self.param_groups:
            scale = g["rho"] / (norm + 1e-12)
            for p in g["params"]:
                if p.grad is not None:
                    e = p.grad * scale
                    p.add_(e)
                    self.state[p]["e"] = e
        self.zero_grad()

    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g["params"]:
                if "e" in self.state[p]:
                    p.sub_(self.state[p].pop("e"))
        self.base.step()
        self.zero_grad()

def bn_frozen(m, frozen):
    """The second SAM pass must not move BatchNorm's running statistics a second time. With FREEZE_BN they never
    move: BatchNorm runs as released (eval mode) while the weights train."""
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            if FREEZE_BN:
                mod.eval()
            else:
                mod.momentum = 0.0 if frozen else 0.03

class EMA:
    """An exponential moving average of the weights, warming up so a short run is not dominated by the start."""
    def __init__(self, m, decay):
        import copy
        self.m, self.decay, self.n = copy.deepcopy(m).eval(), decay, 0
        for p in self.m.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, m):
        self.n += 1
        d = self.decay * (1 - math.exp(-self.n / 50))
        for e, v in zip(self.m.state_dict().values(), m.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(d).add_(v.detach(), alpha=1 - d)

model = fold_to_gray(released())
ema = EMA(model, EMA_DECAY)
opt = SAM(model.parameters(), torch.optim.SGD, rho=RHO, lr=LR, momentum=MOMENTUM, weight_decay=5e-4,
          nesterov=MOMENTUM > 0)
steps = max(1, EPOCHS * math.ceil(len(train) / BATCH))
warm = min(steps, max(10, steps // 5))  # linear warmup, then cosine down: YOLOX's own recipe warms up for 5 epochs
sched = torch.optim.lr_scheduler.LambdaLR(opt.base, lambda i: min(1.0, (i + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(i, steps) / steps)))
skipped = 0
t0, step = time.time(), 0
for epoch in range(EPOCHS):
    model.train()
    order = list(range(len(train))); random.shuffle(order)
    losses = []
    for i in range(0, len(order), BATCH):
        batch = [sample(*train[j], True) for j in order[i:i + BATCH]]
        x, t = torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch])
        bn_frozen(model, False)
        loss = model(x, t)["total_loss"]
        if not torch.isfinite(loss):
            opt.zero_grad(); skipped += 1; continue
        loss.backward()
        opt.first_step()
        bn_frozen(model, True)
        second = model(x, t)["total_loss"]
        if not torch.isfinite(second):  # come back from the perturbed point without stepping
            opt.zero_grad()
            with torch.no_grad():
                for g in opt.param_groups:
                    for p in g["params"]:
                        if "e" in opt.state[p]:
                            p.sub_(opt.state[p].pop("e"))
            skipped += 1; continue
        second.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        opt.second_step()
        ema.update(model)
        sched.step(); step += 1
        losses.append(float(loss.detach()))
    bn_frozen(model, False)
    print(f"3. epoch {epoch + 1}/{EPOCHS}: loss {np.mean(losses) if losses else float('nan'):.2f}, {time.time() - t0:.0f} s so far"
          + (f", {skipped} steps skipped (not finite)" if skipped else ""))

model = ema.m  # the averaged weights are what is scored and exported

# ---------------------------------------------------------------------------------------------- 4. the gate
@torch.no_grad()
def decisions(m, frames, classes):
    """For each frame and class: (labelled present, predicted present at DECISION_SCORE)."""
    m.eval(); m.head.decode_in_inference = True
    truth, said = [], []
    for i in range(0, len(frames), 8):
        chunk = frames[i:i + 8]
        x = torch.stack([sample(p, b, False)[0] for p, b in chunk])
        out = m(x)  # decoded: cx, cy, w, h, objectness, class scores
        for (p, boxes), o in zip(chunk, out):
            score, cls = (o[:, 5:] * o[:, 4:5]).max(1)
            present = {int(c) for c in cls[score >= DECISION_SCORE]}
            have = {b[0] for b in boxes}
            truth.append([c in have for c in classes]); said.append([c in present for c in classes])
    m.head.decode_in_inference = False
    return np.array(truth), np.array(said)

def mcc(t, s):
    t, s = np.asarray(t, bool).ravel(), np.asarray(s, bool).ravel()
    tp, tn, fp, fn = (t & s).sum(), (~t & ~s).sum(), (~t & s).sum(), (t & ~s).sum()
    d = math.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return float((tp * tn - fp * fn) / d) if d else 0.0

EVAL_CLASSES = EVAL_CLASSES or KEEP
classes = [HEAD_ROWS.index(c) for c in EVAL_CLASSES] if EVAL_CLASSES else sorted({b[0] for _, bs in held for b in bs})
base_t, base_s = decisions(fold_to_gray(released()), held, classes)
new_t, new_s = decisions(model, held, classes)
base_mcc, new_mcc = mcc(base_t, base_s), mcc(new_t, new_s)
rows = []
for k, c in enumerate(classes):
    if base_t[:, k].any():
        rows.append((HEAD_ROWS[c], int(base_t[:, k].sum()), mcc(base_t[:, k], base_s[:, k]), mcc(new_t[:, k], new_s[:, k])))
rows.sort(key=lambda r: -r[1])
print(f"4. MCC on {len(held)} held-out frames x {len(classes)} classes (\"is there one?\" at {DECISION_SCORE}):")
print(f"   released model {base_mcc:.3f}  ->  fine-tuned {new_mcc:.3f}")
for name, n, b, f in rows[:8]:
    print(f"   {name:14} in {n:3} frames: {b:.3f} -> {f:.3f}")
# How sure is the gain? Resample the held-out frames with replacement and score both models on each resample.
rng = np.random.default_rng(SEED)
gains = []
for _ in range(1000):
    i = rng.integers(0, len(held), len(held))
    gains.append(mcc(new_t[i], new_s[i]) - mcc(base_t[i], base_s[i]))
lo, hi = np.percentile(gains, [5, 95])
print(f"   gain {new_mcc - base_mcc:+.3f}, 90% bootstrap interval [{lo:+.3f}, {hi:+.3f}] over {len(held)} frames"
      + ("  (too few held-out frames to tell a real gain from chance)" if hi - lo > 0.2 else ""))
drops = [(n, b - f) for n, _, b, f in rows if b - f > MAX_CLASS_DROP]
passed = new_mcc >= base_mcc + MIN_GAIN and new_mcc >= MIN_MCC and not drops and (lo > 0 or not REQUIRE_SURE_GAIN)
print("   GATE:", "PASS" if passed else "FAIL", "" if passed else
      f"(needs MCC >= {base_mcc + MIN_GAIN:.3f} and >= {MIN_MCC}; classes that dropped: {[d[0] for d in drops]})")

# ---------------------------------------------------------------------------------------------- 5. export
def export(m, path):
    m.eval(); m.head.decode_in_inference = False
    torch.onnx.export(m, torch.zeros(1, 1, SIZE, SIZE), str(path), input_names=["images"], output_names=["output"],
                      dynamic_axes={"images": {0: "batch"}, "output": {0: "batch"}}, opset_version=11, dynamo=False)
    return path

import openvino as ov
core = ov.Core()
def run(path, x):
    cm = core.read_model(str(path)); cm.reshape([-1, 1, SIZE, SIZE])
    return core.compile_model(cm, "CPU", {"INFERENCE_PRECISION_HINT": "f32"})({0: x})[0]

# The export path itself, checked: the released model exported this way computes what YOLOX's released ONNX does, folded.
ref_onnx = fetch("https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx",
                 "c789161ed43c8269fcd4e67c67eeeb4e80c622da2eb296a20bc6007bd18a0b7d")
if SAME_HEAD:
    ours = export(fold_to_gray(released()), WORK / "released_gray_check.onnx")
    x = np.random.default_rng(0).integers(0, 256, (2, 1, SIZE, SIZE)).astype(np.float32)
    theirs = core.compile_model(str(ref_onnx), "CPU", {"INFERENCE_PRECISION_HINT": "f32"})
    want = np.concatenate([theirs({0: np.repeat(x[i:i + 1], 3, 1)})[0] for i in range(2)])
    print(f"5. export check: the released weights exported here vs YOLOX's released ONNX: largest difference {np.abs(run(ours, x) - want).max():.1e}")
if passed:
    out = export(model, WORK / "yolox_nano_gray_finetuned.onnx")
    (WORK / "yolox_nano_gray_finetuned.names.json").write_text(json.dumps(HEAD_ROWS))  # detector.names, if new
    x = np.stack([sample(p, b, False)[0].numpy() for p, b in held[:2]])
    with torch.no_grad():
        d = np.abs(run(out, x) - model(torch.from_numpy(x)).numpy()).max()
    print(f"   exported {out.name} (1 channel, any batch); OpenVINO vs PyTorch: largest difference {d:.1e}")
else:
    print("   not exported: the fine-tuned model did not pass the gate. More frames from the sites, more epochs, or a lower LR.")
